# Testing out pyphot

[pyphot docs](https://mfouesneau.github.io/pyphot/index.html)

## Imports

In [13]:
import numpy as np
import matplotlib.pyplot as plt
import pyphot
from pyphot.config import units
from astropy import units as u
from astropy.table import Table

from gaianir_open_clusters.config import DATA_DIRECTORY

## Defining filters

In [2]:
lib = pyphot.get_library()

In [3]:
passbands = lib.load_filters(["Gaia_rvs"])
passbands

[Filter: Gaia_rvs, <pyphot.phot.Filter object at 0x7f9b2e63d010>]

In [4]:
passbands[0].info(show_zeropoints=True)

Filter object information:
    name:                 Gaia_rvs
    detector type:        photon
    wavelength units:     nm
    central wavelength:   860.374755 nm
    pivot wavelength:     860.334135 nm
    effective wavelength: 859.988265 nm
    photon wavelength:    860.061011 nm
    minimum wavelength:   845.603267 nm
    maximum wavelength:   875.554393 nm
    norm:                 24.857119
    effective width:      25.146569 nm
    fullwidth half-max:   0.220324 nm
    definition contains 61 points
    Zeropoints
        Vega: 22.622774 mag,
              8.930797378044035e-10 erg / (Angstrom s cm2),
              2204.9758762194506 Jy
              381.9272030080907 ph / (Angstrom s cm2)
          AB: 22.081284 mag,
              1.4705723425535656e-09 erg / (Angstrom s cm2),
              3630.7805477010015 Jy
          ST: 21.100000 mag,
              3.6307805477010028e-09 erg / (Angstrom s cm2),
              8964.24269932427 Jy
        


In [5]:
gaia_nir_filters = {
    "N": {"mid": 1550.0, "width": 1500},
    "N_R": {"mid": 975, "width": 350},
    "N_J": {"mid": 1275, "width": 250},
    "N_H": {"mid": 1600, "width": 400},
    "N_K": {"mid": 2050, "width": 500},
}

pyphot_filters = {}

for filter, values in gaia_nir_filters.items():
    half_width = values["width"] / 2
    gaia_nir_filters[filter]["wavelengths"] = np.asarray(
        [
            values["mid"] - half_width - 0.0000001,
            values["mid"] - half_width,
            values["mid"],
            values["mid"] + half_width,
            values["mid"] + half_width + 0.0000001,
        ]
    )
    gaia_nir_filters[filter]["transmission"] = np.asarray([0.0] + [0.7] * 3 + [0.0])

    pyphot_filters[filter] = pyphot.Filter(
        gaia_nir_filters[filter]["wavelengths"] * units.U("nm"),
        gaia_nir_filters[filter]["transmission"],
        name=filter,
        dtype="photon",
        vega="stis_011",  # Specify the Vega flavor,
    )

In [ ]:
pyphot_filters['N'].make_integration_filter

np.float64(24.27460110956193)

In [7]:
pyphot_filters['N'].info()

Filter object information:
    name:                 N
    detector type:        photon
    wavelength units:     nm
    central wavelength:   1550.000000 nm
    pivot wavelength:     1443.592716 nm
    effective wavelength: 1156.197873 nm
    photon wavelength:    1252.351412 nm
    minimum wavelength:   800.000000 nm
    maximum wavelength:   2300.000000 nm
    norm:                 1050.000000
    effective width:      1500.000000 nm
    fullwidth half-max:   1500.000000 nm
    definition contains 5 points
    Zeropoints
        Vega: 24.274601 mag,
              1.9505610880307667e-10 erg / (Angstrom s cm2),
              1355.901737543575 Jy
              106.50818482432867 ph / (Angstrom s cm2)
          AB: 23.205172 mag,
              5.223136057303726e-10 erg / (Angstrom s cm2),
              3630.780547701008 Jy
          ST: 21.100000 mag,
              3.6307805477010028e-09 erg / (Angstrom s cm2),
              25238.79761303611 Jy
        


WOW! That is *amazingly* easy :D

## Integrating photometry

In [14]:
file = DATA_DIRECTORY / "models/Kurucz2003all/fm05at3500g00k2odfnew.fl.dat"

In [15]:
Table.read(file)

FileNotFoundError: [Errno 2] No such file or directory: '/home/emily/code/gaianir-open-clusters/data/models/Kurucz2003all/fm05at3500g00k2odfnew.fl.dat'

In [11]:
pyphot_filters['N'].Vega_zero_photons

<Quantity 106.50818482 ph / (Angstrom s cm2)>